# Size scaling — point cloud

Write/read/disk-size of point clouds at increasing `N`, with **CSV as a
baseline** for context. Same `chunk_shape` across runs so the only
variable is vertex count.

For each `N` we measure:

| Operation | zarr-vectors | CSV (baseline) |
| --- | --- | --- |
| Write   | `write_points` | `pandas.to_csv` |
| Read all  | `read_points` | `pandas.read_csv` |
| Read one  | one chunk via lazy API | `read_csv(nrows=1)` (best case) |
| Disk size | store directory | CSV file |

Runtime: a few minutes on a laptop (the 1M case dominates).

In [1]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'

ModuleNotFoundError: No module named 'pandas'

## 1 · Setup

In [ ]:
from zarr_vectors.types.points import write_points, read_points
from zarr_vectors.lazy import open_zv

SIZES = [1_000, 10_000, 100_000, 1_000_000]
CHUNK = (200.0, 200.0, 200.0)
BIN   = (50.0, 50.0, 50.0)
SEED  = 0


def _csv_path(prefix):
    """Fresh tempdir + CSV path."""
    return Path(tempfile.mkdtemp(prefix=f'csvbench_{prefix}_')) / 'points.csv'


def _csv_write(path, positions, intensity):
    """Baseline: write x,y,z,intensity columns to a CSV."""
    pd.DataFrame({
        'x': positions[:, 0],
        'y': positions[:, 1],
        'z': positions[:, 2],
        'intensity': intensity,
    }).to_csv(path, index=False)


def _csv_read_all(path):
    """Read every row back into memory."""
    return pd.read_csv(path)


def _csv_read_one(path):
    """Best-case single-row read: only parse the first data row.

    CSV has no random access, so this is the cheapest single-record
    read the format admits."""
    return pd.read_csv(path, nrows=1)


def _zv_read_one(store_path):
    """Read just one chunk's worth of vertices via the lazy API.

    Touches a single chunk on disk (vs. the full materialisation in
    ``read_points``)."""
    zv = open_zv(store_path)
    chunk_keys = zv[0].vertices._chunk_keys  # noqa: SLF001 — minimal demo
    if not chunk_keys:
        return None
    return zv[0].vertices[chunk_keys[0]].compute()

## 2 · Run the sweep

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for n in SIZES:
    positions = rng.uniform(0, 1000, (n, 3)).astype(np.float32)
    intensity = rng.uniform(0, 1, n).astype(np.float32)

    # ---- ZV ----
    store = _new_store(f'size_{n}')
    t_zv_write, _ = _time(
        write_points, store, positions,
        chunk_shape=CHUNK, bin_shape=BIN,
        attributes={'intensity': intensity},
    )
    t_zv_read_all, _ = _time(read_points, store, attribute_names=['intensity'])
    t_zv_read_one, _ = _time(_zv_read_one, store)
    size_zv_MB = _store_bytes(store) / 1e6

    # ---- CSV baseline ----
    csv = _csv_path(f'size_{n}')
    t_csv_write, _ = _time(_csv_write, csv, positions, intensity)
    t_csv_read_all, _ = _time(_csv_read_all, csv)
    t_csv_read_one, _ = _time(_csv_read_one, csv)
    size_csv_MB = csv.stat().st_size / 1e6

    rows.append({
        'N': n,
        'zv_write_s':    round(t_zv_write,    4),
        'csv_write_s':   round(t_csv_write,   4),
        'zv_read_all_s': round(t_zv_read_all, 4),
        'csv_read_all_s':round(t_csv_read_all,4),
        'zv_read_one_s': round(t_zv_read_one, 4),
        'csv_read_one_s':round(t_csv_read_one,4),
        'zv_size_MB':    round(size_zv_MB,  2),
        'csv_size_MB':   round(size_csv_MB, 2),
    })

    shutil.rmtree(Path(store).parent, ignore_errors=True)
    shutil.rmtree(csv.parent, ignore_errors=True)

df = pd.DataFrame(rows)

## 3 · Results

In [ ]:
df

## 4 · Plot

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), sharex=True)

panels = [
    ('Write time', 'write_s',    'zv_write_s',    'csv_write_s',    's'),
    ('Read all',   'read_all_s', 'zv_read_all_s', 'csv_read_all_s', 's'),
    ('Read one',   'read_one_s', 'zv_read_one_s', 'csv_read_one_s', 's'),
    ('Disk size',  'size_MB',    'zv_size_MB',    'csv_size_MB',    'MB'),
]
for ax, (title, _key, zv_col, csv_col, unit) in zip(axes, panels):
    ax.loglog(df['N'], df[zv_col],  'o-', label='zarr-vectors', color='tab:blue')
    ax.loglog(df['N'], df[csv_col], 's-', label='csv',          color='tab:orange')
    ax.set_title(title)
    ax.set_xlabel('N (vertices)')
    ax.set_ylabel(unit)
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()

fig.suptitle('zarr-vectors vs CSV — point cloud scaling', y=1.02)
plt.tight_layout()